In [1]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 불러오기

In [2]:
df = pd.read_csv("./data/저출산 소개팅_설문조사_시나리오추가.csv")

In [3]:
df.head()

,타임스탬프,0. 당신의 성함,0. 당신의 이상형,1-1. 희망하는 자녀 수,1-2. 희망하는 자녀 구성,1-3. 자녀 갖고 싶은 시기,1-4. 생물학적 출산이 어려움 발생 시 대안,"""1. 자녀 계획 및 가족 구성 항목""에 대해 중요도","아이가 ""오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?",평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.,...,맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?,"양가 어르신들이 본인의 가치관과 다른 육아 조언(예: ""애를 너무 손타게 키운다"", ""사탕 좀 주면 어떠냐"")을 하실 때 당신의 생각은?","주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?",아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?,4-1. 자녀 교육비/양육비 지출 비중,"4-2. 육아 휴직, 양육 부담","""4. 경제적 지원 및 가사 분담""에 대해 중요도","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?","""5. 자녀 가치관""에 대해 중요도",Unnamed: 23
0,2026/02/27 3:53:18 PM GMT+9,강현준,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,...,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3,https://drive.google.com/u/0/open?usp=forms_we...
1,2026/02/27 4:15:15 PM GMT+9,임경빈,꼬부기상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,...,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3,NaN
2,2026/02/27 4:19:32 PM GMT+9,이정미,곰상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,...,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4,NaN
3,2026/02/27 4:29:59 PM GMT+9,쳌,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,...,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2,NaN
4,2026/02/27 4:30:18 PM GMT+9,김용희,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,...,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 24 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   타임스탬프                                                                                  34 non-null     str  
 1   0. 당신의 성함                                                                              34 non-null     str  
 2   0. 당신의 이상형                                                                             34 non-null     str  
 3   1-1. 희망하는 자녀 수                                                                         34 non-null     str  
 4   1-2. 희망하는 자녀 구성                                                                        34 non-null     str  
 5   1-3. 자녀 갖고 싶은 시기                                                                       34 non-null     st

# 2. 데이터 전처리

In [6]:
df_raw = df.copy()

## (1) 컬럼 정리

In [ ]:
# 타임스탬프, 유저이름 - 제거

# ------- 시나리오 기반 설문 문항 --------
#   5개 축(S/F, A/H, E/L, B/T, P/G)에 연결
#   1이면 왼쪽, 5면 오른쪽 규칙을 코드로 고정
#   유저별 타입문자열(value_type)만들기 -> 점수 벡터화(axis_vector)
# -------------------------------------

# ---------- 기타 추천 보완 문항 -----------
#   자녀 계획 및 가족구성, 경제적 지원 및 가사분담, 자녀 가치관  
#   컬럼명 정리 및 원핫인코딩
# -----------------------------------------

# 가중치 문항값을 더해서 유사도, 보완도 추천에 사용

In [7]:
# =========================
# 1) 컬럼명 정규화 함수 (공백/따옴표/줄바꿈 정리)
# =========================
def clean_colname(col: str) -> str:
    """
    - 앞뒤 공백 제거
    - 줄바꿈/탭 등을 공백으로 변경
    - 연속 공백은 1개로 축소
    - 큰따옴표(") 제거  -> 구글폼/CSV 저장 과정에서 섞이는 경우가 많음
    """
    col = str(col).strip()
    col = col.replace("\n", " ").replace("\t", " ")
    col = re.sub(r"\s+", " ", col)      # 공백 여러 개 -> 하나
    col = col.replace('"', "")          # 큰따옴표 제거
    return col

# =========================
# 2) 정규화 적용 + 중복 컬럼명 방지(매우 중요)
# =========================
cleaned_cols = [clean_colname(c) for c in df.columns]

def make_unique(names):
    """
    정규화 후 같은 이름이 중복되면 모델링/전처리에서 에러가 나므로
    자동으로 _1, _2 같은 접미사를 붙여 유니크하게 만든다.
    """
    seen = {}
    out = []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}_{seen[n]}")
    return out

df.columns = make_unique(cleaned_cols)

# =========================
# 3) Unnamed 같은 잔재 컬럼 제거 + (선택) 거의 빈 컬럼 제거
# =========================
# 3-1) 이름이 'Unnamed'로 시작하는 컬럼 제거
unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

# 3-2) (선택) non-null이 1개 이하인 컬럼 제거 후보
#       지금은 Unnamed만 제거해도 충분하지만, 안전하게 후보를 확인해볼 수 있다.
low_nonnull_cols = [c for c in df.columns if df[c].notna().sum() <= 1]
# Unnamed는 이미 제거되었을 가능성이 높음. 혹시 남아있으면 같이 제거.
low_nonnull_cols = [c for c in low_nonnull_cols if c not in unnamed_cols]

# 너무 공격적으로 제거하고 싶지 않다면 "출력만" 하고 넘어가도 된다.
if low_nonnull_cols:
    print("[확인] non-null 1개 이하 컬럼 후보:", low_nonnull_cols)
    # 정말 제거할 거면 아래 주석 해제
    # df = df.drop(columns=low_nonnull_cols)

# =========================
# 4) 짧은 컬럼명으로 rename (핵심)
#    - 길고 한국어 문장인 컬럼명은 코드에서 다루기 매우 불편함
#    - 대신, 원문 컬럼명을 '매핑표'로 따로 남겨서 해석 가능하게 한다.
# =========================
rename_map = {
    "타임스탬프": "timestamp",
    "0. 당신의 성함": "user_name",
    "0. 당신의 이상형": "ideal_type",

    "1-1. 희망하는 자녀 수": "plan_children_count",
    "1-2. 희망하는 자녀 구성": "plan_children_composition",
    "1-3. 자녀 갖고 싶은 시기": "plan_children_timing",
    "1-4. 생물학적 출산이 어려움 발생 시 대안": "plan_infertility_alternative",

    # 따옴표 제거(clean_colname) 후의 이름을 기준으로 매칭됨
    "1. 자녀 계획 및 가족 구성 항목에 대해 중요도": "importance_family_plan",

    # 시나리오(정수형) 문항들
    '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?': "scenario_toothbrushing",
    "평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.": "scenario_bedtime_story",
    "경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?": "scenario_competition_2nd",
    "재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?": "scenario_talent_education",
    "두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?": "scenario_discipline_conflict",
    "한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.": "scenario_play_vs_chores",
    "맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?": "scenario_grandparents_help",
    # 따옴표는 clean_colname에서 제거되므로, 예시 문장 안 따옴표도 없는 버전으로 매칭
    "양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?": "scenario_inlaws_advice",
    "주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?": "scenario_rainy_zoo",
    "아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?": "scenario_education_fund_risk",

    "4-1. 자녀 교육비/양육비 지출 비중": "econ_spending_ratio",
    "4-2. 육아 휴직, 양육 부담": "econ_parental_leave_burden",
    "4. 경제적 지원 및 가사 분담에 대해 중요도": "importance_econ_support",

    "5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?": "child_values_open",
    "5. 자녀 가치관에 대해 중요도": "importance_child_values",
}

# rename 시도하기 전에 "매칭이 안 되는 컬럼"이 있는지 점검
missing_in_df = [k for k in rename_map.keys() if k not in df.columns]
if missing_in_df:
    print("\n[주의] rename_map에 있는데 df.columns에서 못 찾은 원문 컬럼명들이 있어요.")
    print("-> 보통 컬럼명에 공백/특수문자 차이로 생깁니다. 아래 목록 확인:")
    for m in missing_in_df:
        print(" -", m)

# 실제 rename 적용 (존재하는 것만 rename)
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# =========================
# 5) 타입 정리 (최소한으로만)
# =========================
# timestamp는 추천 시스템에서 시계열을 쓰지 않아도, 기록/정렬에 유용하므로 datetime으로 바꿔둔다.
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# =========================
# 6) 컬럼 그룹 만들기 (다음 단계 전처리/벡터화에서 매우 유용)
# =========================
META_COLS = [c for c in ["timestamp", "user_name", "ideal_type"] if c in df.columns]

PLAN_COLS = [c for c in [
    "plan_children_count",
    "plan_children_composition",
    "plan_children_timing",
    "plan_infertility_alternative"
] if c in df.columns]

IMPORTANCE_COLS = [c for c in [
    "importance_family_plan",
    "importance_econ_support",
    "importance_child_values"
] if c in df.columns]

ECON_COLS = [c for c in [
    "econ_spending_ratio",
    "econ_parental_leave_burden"
] if c in df.columns]

TEXT_COLS = [c for c in ["child_values_open"] if c in df.columns]

SCENARIO_COLS = [c for c in df.columns if c.startswith("scenario_")]

print("\n==== 컬럼 그룹 요약 ====")
print("META:", META_COLS)
print("PLAN:", PLAN_COLS)
print("IMPORTANCE:", IMPORTANCE_COLS)
print("ECON:", ECON_COLS)
print("TEXT:", TEXT_COLS)
print("SCENARIO:", SCENARIO_COLS)

# =========================
# 7) 최종 점검 (추천/학습 파이프라인 들어가기 전에 꼭)
# =========================
print("\n==== 최종 df 요약 ====")
print("shape:", df.shape)
print("dtypes:\n", df.dtypes)

# 시나리오 값이 정말 정수/범위가 합리적인지 빠르게 확인
if SCENARIO_COLS:
    print("\n==== 시나리오 응답 값 분포(기본) ====")
    print(df[SCENARIO_COLS].describe())


==== 컬럼 그룹 요약 ====
META: ['timestamp', 'user_name', 'ideal_type']
PLAN: ['plan_children_count', 'plan_children_composition', 'plan_children_timing', 'plan_infertility_alternative']
IMPORTANCE: ['importance_family_plan', 'importance_econ_support', 'importance_child_values']
ECON: ['econ_spending_ratio', 'econ_parental_leave_burden']
TEXT: ['child_values_open']
SCENARIO: ['scenario_toothbrushing', 'scenario_bedtime_story', 'scenario_competition_2nd', 'scenario_talent_education', 'scenario_discipline_conflict', 'scenario_play_vs_chores', 'scenario_grandparents_help', 'scenario_inlaws_advice', 'scenario_rainy_zoo', 'scenario_education_fund_risk']

==== 최종 df 요약 ====
shape: (34, 23)
dtypes:
 timestamp                       datetime64[us, UTC-09:00]
user_name                                             str
ideal_type                                            str
plan_children_count                                   str
plan_children_composition                             str
plan_children

C:\Users\user\AppData\Local\Temp\ipykernel_12740\2925777419.py:115: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


## (2) mbti형 컬럼 생성

In [ ]:
# ==========================================
# 0) 시나리오 컬럼을 "df 컬럼 등장 순서대로" 가져오기
#    - 사용자가 말한 "차례로 두개씩 묶기"를 그대로 구현하려면
#      정렬(sort)이 아니라 df.columns 순서를 그대로 써야 합니다.
# ==========================================
scenario_cols = [c for c in df.columns if c.startswith("scenario_")]

print("찾은 scenario_ 컬럼 개수:", len(scenario_cols))
print("scenario_ 컬럼들(등장 순서):")
for i, c in enumerate(scenario_cols, 1):
    print(f"{i:02d}. {c}")

# 안전장치: 이번 설계는 10개(= 5축 * 2문항)를 기대합니다.
if len(scenario_cols) < 10:
    raise ValueError(
        f"scenario_ 컬럼이 10개보다 적습니다. (현재 {len(scenario_cols)}개)\n"
        f"컬럼명 리네임이 누락되었거나, df에 시나리오 문항이 덜 포함되었을 수 있습니다."
    )

# 혹시 10개 초과면? -> '처음 10개만' 사용 (필요하면 확장 가능)
scenario_cols = scenario_cols[:10]

# ==========================================
# 1) "두 개씩" 묶기: (0,1), (2,3), (4,5), (6,7), (8,9)
# ==========================================
pairs = list(zip(scenario_cols[0::2], scenario_cols[1::2]))

# ==========================================
# 2) 5개 축 정의 (왼쪽 글자, 오른쪽 글자, 축 이름 key)
#    - key는 새로 만들 컬럼명에 사용됩니다.
# ==========================================
axes = [
    ("S", "F", "axis_SF"),  # 양육 실행 스타일
    ("A", "H", "axis_AH"),  # 교육/성장 우선순위
    ("E", "L", "axis_EL"),  # 공동양육 운영 방식
    ("B", "T", "axis_BT"),  # 확장가족/경계
    ("P", "G", "axis_PG"),  # 계획/리스크 대응
]

# 축 개수와 pair 개수가 맞는지 확인
if len(pairs) != len(axes):
    raise ValueError(f"pairs({len(pairs)})와 axes({len(axes)}) 개수가 맞지 않습니다.")

# ==========================================
# 3) 타이브레이크 포함: "무조건 한 쪽 글자"를 반환하는 함수
# ==========================================
def pick_letter_no_neutral(v1: float, v2: float, left: str, right: str) -> str:
    """
    v1, v2: 1~5 응답값(결측이면 이미 3으로 채운 뒤 들어오게 처리)
    left/right: 축의 왼쪽/오른쪽 글자

    - signed 변환: (값 - 3)
    - 합계가 양수면 right, 음수면 left
    - 합계가 0이면(중립 가능) 타이브레이크로 중립 제거
    """
    s1 = v1 - 3
    s2 = v2 - 3
    total = s1 + s2

    if total > 0:
        return right
    if total < 0:
        return left

    # ---- 여기부터가 "중립 방지" 타이브레이크 ----
    # 1) 두 번째 문항이 3이 아니면, 그 문항 방향으로 결정
    if s2 > 0:
        return right
    if s2 < 0:
        return left

    # 2) 첫 번째 문항이 3이 아니면, 그 문항 방향으로 결정
    if s1 > 0:
        return right
    if s1 < 0:
        return left

    # 3) 둘 다 정확히 3이면 완전 중립이므로 기본값으로 한쪽 고정
    #    기본값은 오른쪽으로 두었습니다(원하면 왼쪽으로 바꿔도 됨).
    return right


# ==========================================
# 4) 축별 점수/글자 컬럼 생성
#    - axis_*_mean : 두 문항 평균(1~5 범위, float)
#    - axis_*_sum  : 두 문항 합(2~10 범위, int)
#    - axis_*_letter : 최종 글자(중립 없음)
# ==========================================
for (left, right, axis_key), (col1, col2) in zip(axes, pairs):

    # ---- 4-1) 값이 숫자형이 아닐 수 있으니 안전하게 숫자로 변환 ----
    # errors="coerce" : 숫자로 못 바꾸면 NaN(결측)로 만든다.
    v1 = pd.to_numeric(df[col1], errors="coerce")
    v2 = pd.to_numeric(df[col2], errors="coerce")

    # ---- 4-2) 결측치 처리 ----
    # 결측치는 "중립(3)"으로 채워서 파이프라인이 깨지지 않게 한다.
    # (타입은 타이브레이크 때문에 결국 한쪽 글자로 결정됨)
    v1 = v1.fillna(3)
    v2 = v2.fillna(3)

    # ---- 4-3) 축 점수 만들기 (분석/디버깅용으로 같이 저장) ----
    df[f"{axis_key}_sum"] = (v1 + v2).astype(int)      # 2~10
    df[f"{axis_key}_mean"] = (v1 + v2) / 2.0           # 1~5 (float)

    # ---- 4-4) 축 글자 결정(중립 없음) ----
    # row-wise로 v1, v2를 넣어 pick_letter_no_neutral 실행
    df[f"{axis_key}_letter"] = [
        pick_letter_no_neutral(a, b, left, right)
        for a, b in zip(v1.tolist(), v2.tolist())
    ]

    # ---- 4-5) 어떤 시나리오가 어떤 축에 들어갔는지 로그로 확인 ----
    print(f"\n[{axis_key}] {left} vs {right}")
    print(" - 사용 문항 1:", col1)
    print(" - 사용 문항 2:", col2)

# ==========================================
# 5) 최종 "MBTI형(5축)" 타입 문자열 만들기
#    - 예: FHLTG 같은 5글자 코드 (S/F, A/H, E/L, B/T, P/G 순)
# ==========================================
letter_cols = [f"{axis_key}_letter" for _, _, axis_key in axes]

df["childcare_type_5axis"] = df[letter_cols].astype(str).agg("".join, axis=1)

# ==========================================
# 6) 결과 확인
# ==========================================
print("\n==== 생성된 타입 분포(빈도) ====")
print(df["childcare_type_5axis"].value_counts())

print("\n==== 생성된 컬럼 미리보기 ====")
preview_cols = (
    scenario_cols
    + [f"{k}_sum" for _, _, k in axes]
    + [f"{k}_mean" for _, _, k in axes]
    + letter_cols
    + ["childcare_type_5axis"]
)
print(df[preview_cols].head())

찾은 scenario_ 컬럼 개수: 10
scenario_ 컬럼들(등장 순서):
01. scenario_toothbrushing
02. scenario_bedtime_story
03. scenario_competition_2nd
04. scenario_talent_education
05. scenario_discipline_conflict
06. scenario_play_vs_chores
07. scenario_grandparents_help
08. scenario_inlaws_advice
09. scenario_rainy_zoo
10. scenario_education_fund_risk

[axis_SF] S vs F
 - 사용 문항 1: scenario_toothbrushing
 - 사용 문항 2: scenario_bedtime_story

[axis_AH] A vs H
 - 사용 문항 1: scenario_competition_2nd
 - 사용 문항 2: scenario_talent_education

[axis_EL] E vs L
 - 사용 문항 1: scenario_discipline_conflict
 - 사용 문항 2: scenario_play_vs_chores

[axis_BT] B vs T
 - 사용 문항 1: scenario_grandparents_help
 - 사용 문항 2: scenario_inlaws_advice

[axis_PG] P vs G
 - 사용 문항 1: scenario_rainy_zoo
 - 사용 문항 2: scenario_education_fund_risk

==== 생성된 타입 분포(빈도) ====
childcare_type_5axis
FHLTG    5
FHLBG    4
FHEBG    4
FHLBP    3
FHLTP    3
FALTG    2
SHLTG    2
FHETG    2
FHEBP    2
SHEBP    2
SAETP    1
SALBP    1
FHETP    1
FALBP    1
SALTP    

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 39 columns):
 #   Column                        Non-Null Count  Dtype                    
---  ------                        --------------  -----                    
 0   timestamp                     34 non-null     datetime64[us, UTC-09:00]
 1   user_name                     34 non-null     str                      
 2   ideal_type                    34 non-null     str                      
 3   plan_children_count           34 non-null     str                      
 4   plan_children_composition     34 non-null     str                      
 5   plan_children_timing          34 non-null     str                      
 6   plan_infertility_alternative  34 non-null     str                      
 7   importance_family_plan        34 non-null     int64                    
 8   scenario_toothbrushing        34 non-null     int64                    
 9   scenario_bedtime_story        34 non-null     int64     

In [11]:
df.head()

,timestamp,user_name,ideal_type,plan_children_count,plan_children_composition,plan_children_timing,plan_infertility_alternative,importance_family_plan,scenario_toothbrushing,scenario_bedtime_story,...,axis_EL_sum,axis_EL_mean,axis_EL_letter,axis_BT_sum,axis_BT_mean,axis_BT_letter,axis_PG_sum,axis_PG_mean,axis_PG_letter,childcare_type_5axis
0,2026-02-27 15:53:18-09:00,강현준,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,...,8,4.0,L,6,3.0,T,9,4.5,G,FHLTG
1,2026-02-27 16:15:15-09:00,임경빈,꼬부기상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,...,8,4.0,L,8,4.0,T,7,3.5,G,FALTG
2,2026-02-27 16:19:32-09:00,이정미,곰상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,...,9,4.5,L,6,3.0,B,6,3.0,P,FHLBP
3,2026-02-27 16:29:59-09:00,쳌,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,...,8,4.0,L,6,3.0,B,7,3.5,G,FHLBG
4,2026-02-27 16:30:18-09:00,김용희,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,...,5,2.5,E,7,3.5,T,5,2.5,P,SAETP


## (3) 스케일링

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity